# Phase 2b Stage 2: Entity聚类

目标：将每个relation的entities聚类到正好15个

策略：
- 对entities > 15的relations: 用Agglomerative Clustering直接聚成15类
- 对entities ≤ 15的relations: 直接跳过（留给下一阶段LLM微调）
- 无需grid search，直接指定n_clusters=15

In [ ]:
import sys
sys.path.insert(0, '..')

from src.clustering.entity_clusterer import EntityClusterer
from sklearn.cluster import AgglomerativeClustering
from collections import Counter
import json

## 1. 加载数据并统计

In [ ]:
# 初始化
clusterer = EntityClusterer(
    stage1_5_result_path='../results/entity_redistribution_stage1_5_redistributed.json'
)

# 加载Stage 1.5数据
stage1_5_data = clusterer.load_stage1_5_results()

# 统计每个relation的entity数量
print("Relation Entity Counts:")
print("="*80)

relations_to_cluster = []
relations_to_skip = []

for relation in sorted(clusterer.entities_by_relation.keys()):
    n_entities = len(clusterer.entities_by_relation[relation])
    
    if n_entities > 15:
        status = "CLUSTER"
        relations_to_cluster.append(relation)
    else:
        status = "SKIP"
        relations_to_skip.append(relation)
    
    print(f"{relation:30s}: {n_entities:3d} entities [{status}]")

print(f"\n需要聚类: {len(relations_to_cluster)} relations")
print(f"直接跳过: {len(relations_to_skip)} relations ({', '.join(relations_to_skip)})")

## 2. 聚类函数（Agglomerative Clustering）

直接聚类到15个clusters

In [ ]:
# 简化：直接用Agglomerative Clustering聚成15类
from sklearn.cluster import AgglomerativeClustering

def cluster_to_target_size(relation, embeddings, entities, target_size=15):
    """
    直接聚类到指定数量
    
    Args:
        relation: Relation名称
        embeddings: Entity embeddings
        entities: Entity列表
        target_size: 目标聚类数（默认15）
    
    Returns:
        labels: 聚类标签
    """
    print(f"\n{'='*80}")
    print(f"Clustering: {relation} ({len(entities)} entities → {target_size} clusters)")
    print(f"{'='*80}")
    
    # Agglomerative Clustering with cosine distance
    clustering = AgglomerativeClustering(
        n_clusters=target_size,
        metric='cosine',
        linkage='average'
    )
    
    labels = clustering.fit_predict(embeddings)
    
    # 统计每个cluster的大小
    from collections import Counter
    cluster_sizes = Counter(labels)
    max_size = max(cluster_sizes.values())
    min_size = min(cluster_sizes.values())
    avg_size = len(entities) / target_size
    
    print(f"\n✓ 聚类完成:")
    print(f"  Clusters: {target_size}")
    print(f"  每个cluster平均: {avg_size:.1f} entities")
    print(f"  最大cluster: {max_size} entities")
    print(f"  最小cluster: {min_size} entities")
    
    # 显示每个cluster的entities
    print(f"\n  Cluster分布:")
    for cluster_id in sorted(cluster_sizes.keys()):
        cluster_entities = [entities[i] for i, label in enumerate(labels) if label == cluster_id]
        print(f"    Cluster {cluster_id:2d}: {cluster_sizes[cluster_id]:2d} entities - {', '.join(sorted(cluster_entities)[:3])}")
    
    return labels

print("直接聚类到15个clusters，无需grid search")

## 3. 处理所有relations（直接聚15类）

In [ ]:
all_entity_mappings = {}

for idx, relation in enumerate(sorted(clusterer.entities_by_relation.keys()), 1):
    n_entities = len(clusterer.entities_by_relation[relation])
    
    print(f"\n\n{'#'*80}")
    print(f"[{idx}/15] Processing: {relation} ({n_entities} entities)")
    print(f"{'#'*80}")
    
    # 跳过小于等于15的
    if n_entities <= 15:
        print(f"⏭️  Skipping (already ≤ 15 entities)")
        all_entity_mappings[relation] = {
            ent: ent for ent in clusterer.entities_by_relation[relation]
        }
        continue
    
    # 生成embeddings
    embeddings, entities = clusterer.embed_entities_bge(relation)
    
    # 直接聚成15类
    labels = cluster_to_target_size(relation, embeddings, entities, target_size=15)
    
    # 生成entity mapping（按字母顺序选择canonical name）
    entity_mapping = {}
    for cluster_id in set(labels):
        cluster_entities = [entities[i] for i, label in enumerate(labels) if label == cluster_id]
        canonical_name = sorted(cluster_entities)[0]  # 字母序第一个
        
        for entity in cluster_entities:
            entity_mapping[entity] = canonical_name
    
    all_entity_mappings[relation] = entity_mapping
    
    # 简要统计
    n_after = len(set(entity_mapping.values()))
    print(f"\n✅ {relation}: {n_entities} → {n_after} entities")

## 4. 最终统计

In [ ]:
print(f"\n{'='*80}")
print("Final Statistics")
print(f"{'='*80}")

for relation in sorted(all_entity_mappings.keys()):
    mapping = all_entity_mappings[relation]
    n_before = len(mapping)
    n_after = len(set(mapping.values()))
    compression = n_after / n_before if n_before > 0 else 1.0
    
    status = "✓" if n_after <= 15 else "✗"
    print(f"  {status} {relation:30s}: {n_before:3d} → {n_after:3d} ({compression:.1%})")

total_before = sum(len(m) for m in all_entity_mappings.values())
total_after = sum(len(set(m.values())) for m in all_entity_mappings.values())

print(f"\n  {'TOTAL':32s}: {total_before:3d} → {total_after:3d} ({total_after/total_before:.1%})")

# 检查是否所有relations都 <= 15
all_meet_target = all(len(set(m.values())) <= 15 for m in all_entity_mappings.values())
if all_meet_target:
    print("\n🎉 所有relations都满足 ≤15 entities的目标！")
else:
    failed = [r for r, m in all_entity_mappings.items() if len(set(m.values())) > 15]
    print(f"\n⚠️  以下relations未达到目标: {', '.join(failed)}")

## 5. 保存结果

In [ ]:
# 保存entity mappings
clusterer.save_results(
    all_entity_mappings=all_entity_mappings,
    output_path='../results/entity_mapping_agglomerative.json',
    metadata={
        'embedding_model': 'BAAI/bge-base-en-v1.5',
        'clustering_method': 'Agglomerative Clustering',
        'n_clusters': 15,
        'metric': 'cosine',
        'linkage': 'average',
        'canonical_selection': 'alphabetical',
        'note': 'Direct clustering to 15 entities per relation (for relations with >15 entities)'
    }
)

print("\n✅ Phase 2b Stage 2 完成！")
print(f"结果已保存到: ../results/entity_mapping_agglomerative.json")